# 🎯 Customer Segmentation (RFM + KMeans) & Product Recommendations
This notebook focuses on the modeling core of the **Shopper Spectrum** project. We will:
1. Calculate RFM (Recency, Frequency, Monetary) metrics per customer.
2. Preprocess features using the optimized **Log-Transform + StandardScaler** pipeline.
3. Train and tune K-Means clustering ($K=4$), and dynamically map segments.
4. Build an item-based collaborative filtering recommender using Cosine Similarity.
5. Export models and similarity dictionaries to the `models/` directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

# Set styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
import warnings
warnings.filterwarnings('ignore')

## 1. Data Cleaning and Loading
We load the transaction dataset from `../data/online_retail.csv` and filter out missing values, returns, and cancellations.

In [ ]:
csv_path = '../data/online_retail.csv'
df = pd.read_csv(csv_path)

# Clean
clean_df = df.dropna(subset=['CustomerID']).copy()
clean_df = clean_df[~clean_df['InvoiceNo'].astype(str).str.startswith('C')]
clean_df = clean_df[(clean_df['Quantity'] > 0) & (clean_df['UnitPrice'] > 0)]
clean_df['CustomerID'] = clean_df['CustomerID'].astype(int).astype(str)
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
clean_df['TotalAmount'] = clean_df['Quantity'] * clean_df['UnitPrice']
clean_df['Description'] = clean_df['Description'].str.strip().str.upper()

print(f"Cleaned dataframe contains {clean_df.shape[0]:,} rows for {clean_df['CustomerID'].nunique():,} customers.")

## 2. RFM Feature Engineering
We engineer three critical customer behavior features:
* **Recency ($R$):** Days since the customer's last transaction (relative to the dataset max date + 1 day).
* **Frequency ($F$):** Total number of unique transactions (invoices) completed.
* **Monetary ($M$):** Total dollar amount spent.

In [ ]:
# Establish reference date
ref_date = clean_df['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Reference date:", ref_date)

# Calculate RFM metrics
rfm = clean_df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (ref_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

print("RFM Table Head:")
display(rfm.head())
print("\nSummary Statistics:")
display(rfm.describe())

## 3. Log-Transformation and Standardization
Due to severe right skewness in frequency and monetary value, standardizing raw features directamente drags centroids towards high-value outliers and compresses the rest of the customer base. We apply a `log(x + 1)` transform to resolve the skewness, followed by `StandardScaler` to bring features to a standard scale.

In [ ]:
# Log Transform to normalize distribution
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])

# Scale features
scaler_log = StandardScaler()
rfm_log_scaled = pd.DataFrame(
    scaler_log.fit_transform(rfm_log), 
    columns=['Recency', 'Frequency', 'Monetary']
)

# Plot Log-transformed distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, col in enumerate(['Recency', 'Frequency', 'Monetary']):
    sns.histplot(rfm_log[col], kde=True, color='darkgreen', ax=axes[idx])
    axes[idx].set_title(f"Log-Transformed {col}\nSkewness: {rfm_log[col].skew():.2f}")
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning: Elbow Method & Silhouette Score
We evaluate K-Means performance for $K \in [2, 8]$ using Inertia (within-cluster sum of squares) and Silhouette Score (computed on a representative sample of 2,000 customers for efficiency).

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_log_scaled)
    inertia.append(km.inertia_)
    
    # For speed, compute Silhouette score on a random sample of 2000 customers
    sample_idx = rfm_log_scaled.sample(n=2000, random_state=42).index
    silhouette_scores.append(silhouette_score(rfm_log_scaled.loc[sample_idx], labels[sample_idx]))

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(list(K_range), inertia, 'b-x', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (Sum of squared distances)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_title('Elbow Curve & Silhouette Score Analysis for Log-Scaled KMeans')

ax2 = ax1.twinx()
ax2.plot(list(K_range), silhouette_scores, 'r-o', linewidth=2, markersize=8)
ax2.set_ylabel('Silhouette Score (Higher is Better)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

## 5. Final KMeans Model ($K=4$) & Segment Characterization
We fit our final model with $K=4$ and dynamically map the clusters to these standard segment names:
* **High-Value:** Very recent purchases, high transaction frequency, and high monetary spend.
* **Regular:** Frequent, steady spenders with moderate recency.
* **Occasional:** Low transaction frequency, moderate monetary spend, and low days since last purchase.
* **At-Risk:** Lapsed buyers who haven't shopped in a long time, with low frequency and low spend.

In [ ]:
final_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = final_kmeans.fit_predict(rfm_log_scaled)

# Profile each cluster
profile = rfm.groupby('Cluster').agg(
    Recency_Mean=('Recency', 'mean'),
    Frequency_Mean=('Frequency', 'mean'),
    Monetary_Mean=('Monetary', 'mean'),
    Customer_Count=('Cluster', 'count')
).reset_index()

display(profile)

In [ ]:
# Dynamic Mapper based on cluster centroids
highest_monetary_cluster = profile.loc[profile['Monetary_Mean'].idxmax(), 'Cluster']
highest_recency_cluster = profile.loc[profile['Recency_Mean'].idxmax(), 'Cluster']

remaining_clusters = [c for c in [0,1,2,3] if c not in [highest_monetary_cluster, highest_recency_cluster]]

if profile.loc[profile['Cluster'] == remaining_clusters[0], 'Monetary_Mean'].values[0] > \
   profile.loc[profile['Cluster'] == remaining_clusters[1], 'Monetary_Mean'].values[0]:
    regular_cluster = remaining_clusters[0]
    occasional_cluster = remaining_clusters[1]
else:
    regular_cluster = remaining_clusters[1]
    occasional_cluster = remaining_clusters[0]

mapping = {
    highest_monetary_cluster: 'High-Value',
    regular_cluster: 'Regular',
    occasional_cluster: 'Occasional',
    highest_recency_cluster: 'At-Risk'
}

rfm['Segment'] = rfm['Cluster'].map(mapping)
print("Cluster Mapping:", mapping)
print("\nSegment counts:")
print(rfm['Segment'].value_counts())

### 3D Visualization of Customer Segments
Let's visualize the customer segments in 3D space using Matplotlib.

In [ ]:
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

colors = {'High-Value': 'gold', 'Regular': 'dodgerblue', 'Occasional': 'limegreen', 'At-Risk': 'crimson'}

for segment, color in colors.items():
    subset = rfm[rfm['Segment'] == segment]
    ax.scatter(
        np.log1p(subset['Recency']),
        np.log1p(subset['Frequency']),
        np.log1p(subset['Monetary']),
        c=color, label=segment, s=40, alpha=0.6, edgecolors='w', linewidth=0.5
)

ax.set_xlabel('Log(Recency)')
ax.set_ylabel('Log(Frequency)')
ax.set_zlabel('Log(Monetary)')
ax.set_title('3D Representation of Log-Transformed RFM Customer Segments')
ax.legend(title='Customer Segment')
plt.tight_layout()
plt.show()

## 6. Product Recommendation System (Item-Based Collaborative Filtering)
To recommend products, we build a **Customer-Product Interaction Matrix**:
1. Create a pivot table where rows are `CustomerID`, columns are product `Description`, and cell values are binary indicators (`1` if purchased, `0` otherwise).
2. Compute the **Cosine Similarity** between the columns (products).
3. Find the top 5 most similar products for a user-input product name.

In [ ]:
# Create customer-product pivot table with binary indicator
pivot_df = clean_df.pivot_table(
    index='CustomerID', columns='Description', values='Quantity', 
    aggfunc='sum', fill_value=0
)
pivot_binary = pivot_df.clip(upper=1)

print("Interaction matrix shape:", pivot_binary.shape)

In [ ]:
# Compute Item similarity matrix
item_similarity = cosine_similarity(pivot_binary.T)
similarity_df = pd.DataFrame(item_similarity, index=pivot_binary.columns, columns=pivot_binary.columns)
print("Item similarity matrix computed.")

In [ ]:
# Let's define the recommendation function
def get_top_5_recommendations(product_name):
    product_name = product_name.upper().strip()
    if product_name not in similarity_df.index:
        # Simple search for closest matches
        matches = [idx for idx in similarity_df.index if product_name in idx]
        if not matches:
            return f"Product '{product_name}' not found in catalog."
        else:
            print(f"Product not found. Did you mean: {matches[:3]}?")
            product_name = matches[0]
            print(f"Recommending for closest match: '{product_name}'\n")
            
    # Get top 5 similarities (excluding itself)
    similar_items = similarity_df[product_name].sort_values(ascending=False).iloc[1:6]
    return pd.DataFrame({'Product': similar_items.index, 'Similarity Score': similar_items.values})

In [ ]:
# Test recommendation for a popular item
get_top_5_recommendations('WHITE HANGING HEART T-LIGHT HOLDER')

## 7. Model Export & Similarity Dictionary Optimization
We save standardizer, KMeans model, cluster mapping, and optimized top-N recommendations dictionary to the `../models/` directory.

In [ ]:
# Create the models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# 1. Save standardizer and clustering model
joblib.dump(scaler_log, '../models/scaler.pkl')
joblib.dump(final_kmeans, '../models/kmeans_model.pkl')

# 2. Save cluster to label mapping
joblib.dump(mapping, '../models/cluster_mapping.pkl')

# 3. Compute and save top 10 similarity dictionary
similarity_dict = {}
for col in similarity_df.columns:
    top_10 = similarity_df[col].sort_values(ascending=False).iloc[1:11]
    similarity_dict[col] = list(zip(top_10.index, top_10.values.round(4)))

joblib.dump(similarity_dict, '../models/product_similarity.pkl')
print("All modeling artifacts successfully serialized to the '../models/' directory.")